# 02. NumPy 핵심 — pandas 와 OpenCV 가 올라타 있는 바닥

pandas 의 DataFrame 도, OpenCV 의 이미지도 결국 NumPy 배열이다. 여기가 흔들리면 위층에서
이유를 모르는 버그를 만난다.

## 학습 목표
1. `shape`·`dtype`·축(axis) 읽기
2. 슬라이스는 **뷰(view)** — 원본이 같이 바뀐다
3. 브로드캐스팅 규칙
4. 벡터화 vs 파이썬 루프 — 실제로 몇 배인지 측정
5. 불리언 마스킹 (pandas 필터링의 원형)
6. 난수: `np.random.default_rng` (구식 `np.random.seed` 와의 차이)
7. **ragged 배열 함정** — 이 저장소의 옛 OpenCV 코드가 지금 죽는 바로 그 원인

In [1]:
import numpy as np

print("numpy", np.__version__)

numpy 2.5.2


## 1. 배열 만들기와 들여다보기

In [2]:
a = np.array([[1, 2, 3], [4, 5, 6]])
print("a =\n", a)
print("shape", a.shape, "| ndim", a.ndim, "| dtype", a.dtype, "| size", a.size)
print("zeros(2,3)\n", np.zeros((2, 3)))
print("arange(0,10,3)", np.arange(0, 10, 3))
print("linspace(0,1,5)", np.linspace(0, 1, 5))

a =
 [[1 2 3]
 [4 5 6]]
shape (2, 3) | ndim 2 | dtype int64 | size 6
zeros(2,3)
 [[0. 0. 0.]
 [0. 0. 0.]]
arange(0,10,3) [0 3 6 9]
linspace(0,1,5) [0.   0.25 0.5  0.75 1.  ]


`dtype` 은 조용히 결과를 바꾼다. 정수 배열에 소수를 넣으면 **버림**이 일어난다.

In [3]:
ints = np.array([1, 2, 3])
ints[0] = 9.9          # 경고 없이 9 로 잘린다
print(ints, ints.dtype)

floats = ints.astype(np.float64)
floats[0] = 9.9
print(floats, floats.dtype)

[9 2 3] int64
[9.9 2.  3. ] float64


## 2. 슬라이스는 복사가 아니라 뷰다

파이썬 리스트 슬라이스는 복사본이지만, **NumPy 슬라이스는 원본 메모리를 가리키는 창**이다.
pandas 의 `SettingWithCopyWarning` 도 뿌리가 여기다.

In [4]:
base = np.arange(12).reshape(3, 4)
view = base[:2, :2]
view[:] = 0                      # 원본이 바뀐다

print("base =\n", base)
print("view.base is base :", view.base is base)

copied = base[:2, :2].copy()
copied[:] = 99
print("copy 수정 후 base 는 그대로 =\n", base)

base =
 [[ 0  0  2  3]
 [ 0  0  6  7]
 [ 8  9 10 11]]
view.base is base : False
copy 수정 후 base 는 그대로 =
 [[ 0  0  2  3]
 [ 0  0  6  7]
 [ 8  9 10 11]]


## 3. 브로드캐스팅

모양이 다른 배열끼리 연산할 때 NumPy 는 **뒤 축부터 맞춰 본다**. 각 축은
크기가 같거나, 둘 중 하나가 1 이어야 한다.

In [5]:
matrix = np.arange(12).reshape(3, 4)
row = np.array([10, 20, 30, 40])          # (4,)   → (1,4) 로 늘어남
column = np.array([[100], [200], [300]])  # (3,1)  → (3,4) 로 늘어남

print("matrix + row =\n", matrix + row)
print("matrix + column =\n", matrix + column)

try:
    matrix + np.array([1, 2, 3])          # (3,) 는 마지막 축 4 와 안 맞는다
except ValueError as exc:
    print("\nValueError:", exc)

matrix + row =
 [[10 21 32 43]
 [14 25 36 47]
 [18 29 40 51]]
matrix + column =
 [[100 101 102 103]
 [204 205 206 207]
 [308 309 310 311]]

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


오류 메시지의 `(3,4) (3,)` 를 보고 **어느 축이 안 맞는지** 읽는 습관을 들이면
디버깅 시간이 크게 준다. 해결은 보통 `reshape(-1, 1)` 로 축을 하나 세우는 것이다.

In [6]:
print((matrix + np.array([1, 2, 3]).reshape(-1, 1)).shape)

(3, 4)


## 4. 벡터화 — 얼마나 빠른가

"느리면 루프를 지워라"는 말은 자주 듣지만, 몇 배인지 재 본 사람은 드물다.

In [7]:
size = 1_000_000
data = np.random.default_rng(0).random(size)
pure = data.tolist()

In [8]:
%%timeit -n 3 -r 3
total = 0.0
for value in pure:      # 순수 파이썬 루프
    total += value * value

49.5 ms ± 1.23 ms per loop (mean ± std. dev. of 3 runs, 3 loops each)


In [9]:
%%timeit -n 3 -r 3
total = (data * data).sum()   # 벡터화

6.37 ms ± 1.87 ms per loop (mean ± std. dev. of 3 runs, 3 loops each)


`%%timeit` 출력은 다른 작업의 영향을 받으므로, 배수는 직접 재서 확인하는 편이 정확하다.

In [10]:
import timeit

loop_time = min(timeit.repeat(lambda: sum(v * v for v in pure), number=1, repeat=3))
vector_time = min(timeit.repeat(lambda: (data * data).sum(), number=1, repeat=3))
print(f"파이썬 루프 {loop_time * 1000:8.2f} ms")
print(f"벡터화      {vector_time * 1000:8.2f} ms")
print(f"→ {loop_time / vector_time:.1f}배 빠르다 (원소 {size:,}개 기준)")

파이썬 루프    80.61 ms
벡터화          4.67 ms
→ 17.3배 빠르다 (원소 1,000,000개 기준)


차이가 나는 이유는 두 가지다.
* 파이썬 루프는 원소마다 객체를 만들고 타입을 확인한다
* NumPy 는 연속된 메모리 위에서 C 로 한 번에 돈다 (SIMD 도 탄다)

배수는 연산의 종류와 데이터 크기에 따라 몇 배에서 수백 배까지 달라진다. 중요한 것은 방향이다 —
**원소 단위 for 문이 보이면 벡터화할 수 있는지부터 의심한다.**

## 5. 집계와 축(axis)

`axis` 는 "없어지는 축"이라고 외우면 헷갈리지 않는다. `axis=0` 이면 행이 사라지고 열별 결과가 남는다.

In [11]:
scores = np.array([[80, 90, 70], [60, 85, 95], [75, 65, 88]])
print("전체 평균     ", scores.mean().round(2))
print("열별(axis=0)  ", scores.mean(axis=0).round(2), "← 행이 사라짐")
print("행별(axis=1)  ", scores.mean(axis=1).round(2), "← 열이 사라짐")
print("최댓값 위치   ", np.unravel_index(scores.argmax(), scores.shape))

전체 평균      78.67
열별(axis=0)   [71.67 80.   84.33] ← 행이 사라짐
행별(axis=1)   [80. 80. 76.] ← 열이 사라짐
최댓값 위치    (np.int64(1), np.int64(2))


## 6. 불리언 마스킹 — pandas 필터링의 원형

In [12]:
values = np.array([12, 45, 7, 88, 23, 61, 5])
mask = values > 20

print("mask       ", mask)
print("선택        ", values[mask])
print("개수        ", mask.sum(), "  (True=1 이므로 sum 이 곧 개수)")
print("조건부 치환 ", np.where(values > 20, values, 0))
print("두 조건     ", values[(values > 10) & (values < 60)], "  ← and 가 아니라 &, 괄호 필수")

mask        [False  True False  True  True  True False]
선택         [45 88 23 61]
개수         4   (True=1 이므로 sum 이 곧 개수)
조건부 치환  [ 0 45  0 88 23 61  0]
두 조건      [12 45 23]   ← and 가 아니라 &, 괄호 필수


`and`/`or` 는 배열에 쓸 수 없다(`ValueError: truth value of an array ...`).
`&`, `|`, `~` 를 쓰고 **연산자 우선순위 때문에 각 조건을 괄호로 감싼다.**

## 7. 난수 — 구식과 현행

2019년 코드에서 흔한 `np.random.seed(0)` + `np.random.rand()` 는 **전역 상태**를 쓴다.
다른 라이브러리가 같은 전역을 건드리면 재현성이 깨진다. 현행 권장은 `Generator` 다.

In [13]:
legacy = np.random.RandomState(42)          # 구식(전역 대신 객체를 쓰는 그나마 나은 형태)
modern = np.random.default_rng(42)          # 현행 권장

print("RandomState :", legacy.random(3).round(4))
print("Generator   :", modern.random(3).round(4))
print("같은 시드로 다시:", np.random.default_rng(42).random(3).round(4))

RandomState : [0.3745 0.9507 0.732 ]
Generator   : [0.774  0.4389 0.8586]
같은 시드로 다시: [0.774  0.4389 0.8586]


## 8. ragged 배열 — 이 저장소의 옛 코드가 죽는 이유

길이가 서로 다른 배열들을 `np.array()` 로 묶으려 하면 NumPy 1.24 부터 **예외**가 난다.
예전에는 `dtype=object` 배열로 조용히 만들어졌다.

`legacy/cv2 test.ipynb` 의 `contours_xy = np.array(contours)` 가 정확히 이 코드다.
컨투어마다 점 개수가 다르기 때문에 지금 실행하면 다음과 같이 멈춘다.

In [14]:
contour_like = [np.zeros((5, 1, 2), int), np.zeros((8, 1, 2), int), np.zeros((3, 1, 2), int)]

try:
    np.array(contour_like)
except ValueError as exc:
    print("ValueError:", str(exc)[:120], "...")

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detecte ...


해결은 목적에 따라 다르다.

In [15]:
# ① 좌표를 전부 한 덩어리로 합치고 싶다 → vstack (컨투어 전체 바운딩박스 계산 등)
stacked = np.vstack(contour_like)
print("vstack shape:", stacked.shape)

# ② 개별 배열을 그냥 담아 두고 싶다 → 리스트 그대로 쓰거나 dtype=object 를 명시
boxed = np.empty(len(contour_like), dtype=object)
boxed[:] = contour_like
print("object 배열 :", boxed.shape, boxed.dtype)

vstack shape: (16, 1, 2)
object 배열 : (3,) object


## 정리

| 함정 | 대응 |
|---|---|
| 정수 배열에 소수 대입 → 버림 | `astype(float)` 로 먼저 승격 |
| 슬라이스 수정이 원본에 반영 | 필요하면 `.copy()` |
| 브로드캐스팅 실패 | 에러의 두 shape 를 읽고 `reshape(-1,1)` |
| 원소 단위 for 문 | 벡터화 (수십 배) |
| `values > 10 and values < 60` | `(values > 10) & (values < 60)` |
| `np.array(길이가_다른_리스트)` | `np.vstack` 또는 `dtype=object` 명시 |

다음: **03. pandas 핵심**